In [1]:
import kagglehub
import os
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.decomposition import TruncatedSVD

# **Load Data** from Kaggle

In [2]:
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

print("Path to dataset files:", path)

100%|███████████████████████████████████████████████████████████| 228M/228M [02:31<00:00, 1.57MB/s]

Extracting files...


Path to dataset files: /home/eonzs/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [3]:
print(os.listdir(path))

['credits.csv', 'keywords.csv', 'links.csv', 'links_small.csv', 'movies_metadata.csv', 'ratings.csv', 'ratings_small.csv']


In [4]:
# define file path
movies_metadata_path = os.path.join(path, "movies_metadata.csv")
keywords_path = os.path.join(path, "keywords.csv")
credits_path = os.path.join(path, "credits.csv")
links_path = os.path.join(path, "links.csv")
links_small_path = os.path.join(path, "links_small.csv")
ratings_path = os.path.join(path, "ratings.csv")
rating_small_path = os.path.join(path, "ratings_small.csv")

In [5]:
credits = pd.read_csv(credits_path)
keywords = pd.read_csv(keywords_path)
links = pd.read_csv(links_path)
links_small = pd.read_csv(links_small_path)
movies_metadata = pd.read_csv(movies_metadata_path)
ratings = pd.read_csv(ratings_path)
rating_small = pd.read_csv(rating_small_path)

/tmp/ipykernel_63391/1040381343.py:5: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv(movies_metadata_path)


# **Data Understanding**

In [6]:
movies_metadata

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45461,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45462,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0
45463,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,2003-08-01,0.0,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45464,False,NaN,0,[],NaN,227506,tt0008536,en,Satana likuyushchiy,"In a small town live two brothers, one a minis...",...,1917-10-21,0.0,87.0,[],Released,NaN,Satan Triumphant,False,0.0,0.0


In [7]:
credits

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [8]:
keywords

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."
...,...,...
46414,439050,"[{'id': 10703, 'name': 'tragic love'}]"
46415,111109,"[{'id': 2679, 'name': 'artist'}, {'id': 14531,..."
46416,67758,[]
46417,227506,[]


In [9]:
links

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
45838,176269,6209470,439050.0
45839,176271,2028550,111109.0
45840,176273,303758,67758.0
45841,176275,8536,227506.0


In [10]:
links_small

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9120,162672,3859980,402672.0
9121,163056,4262980,315011.0
9122,163949,2531318,391698.0
9123,164977,27660,137608.0


In [11]:
ratings

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556
...,...,...,...,...
26024284,270896,58559,5.0,1257031564
26024285,270896,60069,5.0,1257032032
26024286,270896,63082,4.5,1257031764
26024287,270896,64957,4.5,1257033990


In [12]:
rating_small

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
...,...,...,...,...
99999,671,6268,2.5,1065579370
100000,671,6269,4.0,1065149201
100001,671,6365,4.0,1070940363
100002,671,6385,2.5,1070979663


Kita akan coba menggunakan data yang kecil dulu, sehingga kita akan memanfaatkan `links_small` untuk memfilter `movies_metadata`. Untuk kasus Collaborative Learning dan Content-based Filtering ini, kita akan menggunakan `movies_metadata` dan `ratings_small` sebagai dataset.

In [13]:
movies_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [14]:
movies_metadata.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [15]:
links_small.info()

<class 'pandas.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9125 non-null   int64  
 1   imdbId   9125 non-null   int64  
 2   tmdbId   9112 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 214.0 KB


In [16]:
links_small.isnull().sum()

movieId     0
imdbId      0
tmdbId     13
dtype: int64

In [17]:
rating_small.info()

<class 'pandas.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


# **Data Preparation**
Pada halaman deskripsi dataset [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) di Kaggle, pembuat dataset (Rounak Banik) menjelaskan bahwa data ini diambil (crawled) dari The Movie Database (TMDb).

Karena sumber datanya adalah TMDb, maka id utama (primary key) yang digunakan dalam file `movie_metadata` tersebut secara default adalah ID milik TMDb itu sendiri.

Oleh karena itu kita akan memanfaatkan TMDB ID `tmdbId` sebagai primary key untuk mengintegrasikan ketiga dataset.

## Cleaning data and Convert to integer

In [18]:
links_small = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')
links_small.info()

<class 'pandas.Series'>
Index: 9112 entries, 0 to 9124
Series name: tmdbId
Non-Null Count  Dtype
--------------  -----
9112 non-null   int64
dtypes: int64(1)
memory usage: 142.4 KB


In [19]:
# Convert 'id' to numeric, forcing errors to NaN, then drop NaNs and convert to int
movies_metadata['id'] = pd.to_numeric(movies_metadata['id'], errors='coerce')
movies_metadata = movies_metadata.dropna(subset=['id'])
movies_metadata['id'] = movies_metadata['id'].astype('int')

movies_metadata['id'].dtype

dtype('int64')

In [20]:
# Since 'id' is already numeric, we cast to string to use .str accessor,
# though this will likely return an empty DataFrame since the data is already cleaned.
metadata_trash = movies_metadata[~movies_metadata['id'].astype("str").str.isnumeric()]

print("Remaining non-numeric IDs:", len(metadata_trash))
if len(metadata_trash) > 0:
    print(metadata_trash[["title", "id", "release_date"]])

Remaining non-numeric IDs: 0


Kita akan memanfaatkan `errors='coerce'` untuk mengubah text yang bukan numerik seperti '20-10-1995' menjadi NaN. Lalu, buang baris yang ID-nya NaN (yang tadi gagal diubah)

In [21]:
movies_metadata['id'] = pd.to_numeric(movies_metadata['id'], errors='coerce')

movies_metadata = movies_metadata.dropna(subset=['id'])
movies_metadata['id'] = movies_metadata['id'].astype('int')

## Filtering `movie_metadata` based on `links_small`'s `tmbdId`

In [22]:
movies_metadata_small = movies_metadata[movies_metadata['id'].isin(links_small)]
movies_metadata_small = movies_metadata_small.reset_index(drop=True)

In [23]:
shape_metadata = movies_metadata_small.shape
shape_links = links_small.shape

print(f"Shape of movies_metadata_small: {shape_metadata}")
print(f"Shape of links_small: {shape_links}")

Shape of movies_metadata_small: (9099, 24)
Shape of links_small: (9112,)


Tentunya kita tidak akan menggunakan seluruh fitur dalam movie_metadata, sehingga mari kita cari fitur mana yang akan kita gunakan. Sampai saat ini, dua fitur yang pastik akan kita gunakan:


*   `id`
*   `title`

Sementara tiga fitur yang tidak akan kita gunakan karena jumlah null yang terlalu banyak:


*   `belongs_to_collection`
*   `homepage`

Fitur `tagline` meskipun memiliki banyak null kita akan menggabungkannya dengan `overview`

## Cleaning `tagline` and merge it with `overview`

In [24]:
movies_metadata_small['tagline'] = movies_metadata_small['tagline'].fillna('')
movies_metadata_small['tagline'].isnull().sum()

np.int64(0)

In [25]:
movies_metadata_small['description'] = movies_metadata_small['overview'] + movies_metadata_small['tagline']

## Checking `description` as a new feature

In [26]:
movies_metadata_small['description'].isnull().sum()

np.int64(12)

In [27]:
movies_metadata_small[movies_metadata_small['description'].isnull() == True].count()

adult                    12
belongs_to_collection     0
budget                   12
genres                   12
homepage                  0
id                       12
imdb_id                  12
original_language        12
original_title           12
overview                  0
popularity               12
poster_path              12
production_companies     12
production_countries     12
release_date             12
revenue                  12
runtime                  12
spoken_languages         12
status                   12
tagline                  12
title                    12
video                    12
vote_average             12
vote_count               12
description               0
dtype: int64

In [28]:
movies_metadata_small['description'] = movies_metadata_small['description'].fillna('')
movies_metadata_small['description'].isnull().sum()

np.int64(0)

## Cleaning `genres` feature

In [29]:
movies_metadata_small['genres'] = movies_metadata_small['genres'].fillna('[]').apply(literal_eval)

In [30]:
movies_metadata_small['genres'] = movies_metadata_small['genres'].apply(lambda x: ' '.join([i['name'] for i in x] if isinstance(x, list) else []))

In [31]:
movie = movies_metadata_small[['id', 'title', 'description', 'genres']]
movie

,id,title,description,genres
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy
...,...,...,...,...
9094,315011,Shin Godzilla,From the mind behind Evangelion comes a hit la...,Action Adventure Drama Horror Science Fiction
9095,391698,The Beatles: Eight Days a Week - The Touring Y...,"The band stormed Europe in 1963, and, in 1964,...",Documentary Music
9096,10991,Pokémon: Spell of the Unknown,When Molly Hale's sadness of her father's disa...,Adventure Fantasy Animation Action Family
9097,12600,Pokémon 4Ever: Celebi - Voice of the Forest,"All your favorite Pokémon characters are back,...",Adventure Fantasy Animation Science Fiction Fa...


# **Content-based Filtering**

## Merge `description` with `genres` for cosine similarity

In [32]:
movie['soup'] = movie['description'] + " " + movie['genres']
movie

,id,title,description,genres,soup
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,A family wedding reignites the ancient feud be...
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just when George Banks has recovered from his ...
...,...,...,...,...,...
9094,315011,Shin Godzilla,From the mind behind Evangelion comes a hit la...,Action Adventure Drama Horror Science Fiction,From the mind behind Evangelion comes a hit la...
9095,391698,The Beatles: Eight Days a Week - The Touring Y...,"The band stormed Europe in 1963, and, in 1964,...",Documentary Music,"The band stormed Europe in 1963, and, in 1964,..."
9096,10991,Pokémon: Spell of the Unknown,When Molly Hale's sadness of her father's disa...,Adventure Fantasy Animation Action Family,When Molly Hale's sadness of her father's disa...
9097,12600,Pokémon 4Ever: Celebi - Voice of the Forest,"All your favorite Pokémon characters are back,...",Adventure Fantasy Animation Science Fiction Fa...,"All your favorite Pokémon characters are back,..."


## TF-IDF initiation and `fit_transform` to vector

In [33]:
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(movie['soup'])

In [34]:
tfidf_matrix.shape

(9099, 274073)

In [35]:
tfidf_matrix.todense()

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(9099, 274073))

## Calculating Cosine Similarity

In [36]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

## Create index for recommendation step

In [37]:
indices = pd.Series(movie.index, index=movie['title']).drop_duplicates()

## Get Content Recommendations

In [38]:
def get_content_recommendations(title, cosine_sim=cosine_sim):
    try:
        idx = indices[title]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:6]

        movie_indices = [i[0] for i in sim_scores]
        scores = [i[1] for i in sim_scores]

        results = movie[['title', 'genres']].iloc[movie_indices].copy()
        results['score'] = scores

        return results

    except KeyError:
        return "Film tidak ditemukan di dataset."

In [39]:
judul_film = 'Toy Story'
print(f"\n🎥 Rekomendasi film mirip '{judul_film}':")
get_content_recommendations(judul_film).head()


🎥 Rekomendasi film mirip 'Toy Story':


,title,genres,score
2502,Toy Story 2,Animation Comedy Family,0.254507
7535,Toy Story 3,Animation Family Comedy,0.227903
6193,The 40 Year Old Virgin,Comedy Romance,0.124769
2547,Man on the Moon,Comedy Drama Romance,0.098590
6627,Factory Girl,Drama,0.078619


# **Collaborative Filtering**

In [40]:
user_movie_matrix = rating_small.pivot(index='userId', columns='movieId', values='rating')

In [41]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [42]:
R = user_movie_matrix_filled.values
user_ratings_mean = np.mean(R, axis=1)
R_demeaned = R - user_ratings_mean.reshape(-1, 1) # Normalisasi

In [43]:
n_components = min(50, R.shape[1]-1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_matrix = svd.fit_transform(R_demeaned)

In [44]:
all_user_predicted_ratings = np.dot(svd_matrix, svd.components_) + user_ratings_mean.reshape(-1, 1)

preds_df = pd.DataFrame(all_user_predicted_ratings, columns=user_movie_matrix.columns, index=user_movie_matrix.index)

In [45]:
def recommend_movies_collaborative(user_id, num_recommendations=5):
    if user_id not in preds_df.index:
        return "User ID tidak ditemukan."

    sorted_user_predictions = preds_df.loc[user_id].sort_values(ascending=False)
    user_data = ratings[ratings.userId == user_id]
    already_rated = user_data['movieId'].tolist()

    recommendations = sorted_user_predictions[~sorted_user_predictions.index.isin(already_rated)]
    top_movie_ids = recommendations.head(num_recommendations).index

    return movie[movie['id'].isin(top_movie_ids)]['title']

In [46]:
user_target = 1
print(f"\n👤 Rekomendasi Personal untuk User ID {user_target}:")
recommend_movies_collaborative(user_target).head()


👤 Rekomendasi Personal untuk User ID 1:


1918                Rocky IV
2161            American Pie
2664                My Tutor
5071    The Butterfly Effect
Name: title, dtype: str

### Try Your Own Recommendations
Use the cell below to change the `my_favorite_movie` title or the `target_user_id` to see different results.

In [47]:
# 1. Get Content-based Recommendations (Based on movie similarity)
my_favorite_movie = 'Cars 2'
print(f"Recommendations for '{my_favorite_movie}':")
display(get_content_recommendations(my_favorite_movie))

print("\n" + "="*50 + "\n")

# 2. Get Collaborative Recommendations (Based on user behavior)
target_user_id = 15
print(f"Personalized Recommendations for User ID {target_user_id}:")
display(recommend_movies_collaborative(target_user_id))

Recommendations for 'Cars 2':


,title,genres,score
6422,Cars,Animation Adventure Comedy Family,0.126582
9074,Central Intelligence,Action Comedy,0.080683
5597,Cannonball,Action,0.069697
4758,Once Upon a Time in Mexico,Action,0.063696
6059,Once Upon a Forest,Animation Family Adventure,0.061988




Personalized Recommendations for User ID 15:


1026                                       Cool Hand Luke
2211                              The Thomas Crown Affair
8747    Shriek If You Know What I Did Last Friday the ...
Name: title, dtype: str